[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_80_Planning_and_Decomposition.ipynb)

# Lesson 80 — Planning & Decomposition
### Phase 9 · Lesson 4 · *from "who does this task" to "what ARE the tasks"*

In **Lesson 79** you built a **Router**: it reads one task, classifies it, and
hands it to the best specialist. That answered a powerful question — **who**
should do this task — but it quietly assumed something big:

> **that the request IS one task.**

Real goals aren't. *"Translate the French invoice, then bill the customer for
the total"* is **one sentence hiding three ordered tasks** with a **data
dependency** (you can't bill an amount you haven't computed yet). Ask a router
to route that whole sentence and it can pick **at most one** specialist — the
rest of the goal silently evaporates.

This lesson builds the missing layer: a **Planner** that
1. **decomposes** a goal into sub-tasks,
2. wires them into a **dependency graph** (a DAG) and puts them in a valid
   **order**,
3. **dispatches** each sub-task through the Lesson-79 Router,
4. **threads** each result into the tasks that depend on it, and
5. **assembles** the final answer — replanning any step that fails.

| Lesson | Topic | The question it answers |
|---|---|---|
| L77 | Topologies | how are agents *wired*? |
| L78 | Blackboard | how do agents *share state*? |
| L79 | Routing & handoff | **who** does this task? |
| **L80 — today** | **Planning & decomposition** | **what are the tasks, in what order?** |
| L81 | Reliability & partial failure | what when a step *breaks*? |
| L82 | Phase-9 capstone | ship the orchestration framework |

Everything runs **offline and deterministic** — no API key, no network.

## 0. Setup — the `orchestra` package + a deterministic world

The first cell writes three modules to disk and imports them:

- `orchestra/core.py` — `Message` / `Agent` (Lesson 77)
- `orchestra/router.py` — `Classifier` / `Router` (Lesson 79)
- `orchestra/planner.py` — `Step` / `Plan` / `topo_order` / `layers` / `Planner` (**new today**)

It also builds a tiny deterministic universe of **four specialties**
(`translate`, `math`, `billing`, `code`). Each specialist truly solves **only
its own** kind of task; a **generalist** can solve any single task but costs
**3× the tokens**. This is the same cast as Lesson 79 — we're adding a
conductor above the orchestra, not replacing the players.

In [ ]:
# --- Write the orchestra package (modules embedded as base64 = collision-proof) ---
import os, sys, base64, importlib

BASE = "/content"          # Colab's working dir; swapped to a tempdir only during offline validation
os.makedirs(os.path.join(BASE, "orchestra"), exist_ok=True)

_CORE_B64   = "IyBvcmNoZXN0cmEvY29yZS5weSAgLS0gIG1lc3NhZ2UgKyBhZ2VudCBwcmltaXRpdmVzIChmcm9tIExlc3NvbiA3NykKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBDYWxsYWJsZQoKQGRhdGFjbGFzcwpjbGFzcyBNZXNzYWdlOgogICAgc2VuZGVyOiBzdHIKICAgIHJlY2lwaWVudDogc3RyCiAgICBraW5kOiBzdHIgICAgICAgICAgICAgICAgICMgInRhc2siIHwgInJlc3VsdCIgfCAiaGFuZG9mZiIgfCAuLi4KICAgIGNvbnRlbnQ6IEFueQogICAgbWV0YTogZGljdCA9IGZpZWxkKGRlZmF1bHRfZmFjdG9yeT1kaWN0KQoKY2xhc3MgQWdlbnQ6CiAgICAjIEFuIGFnZW50ID0gb25lIGNhbGxhYmxlICJicmFpbiIgYmVoaW5kIGEgbmFtZS4gYmFja2VuZChuYW1lLCBjb250ZW50KSAtPiAob3V0cHV0LCB0b2tlbnMpCiAgICBkZWYgX19pbml0X18oc2VsZiwgbmFtZTogc3RyLCByb2xlOiBzdHIsIGJhY2tlbmQ6IENhbGxhYmxlKToKICAgICAgICBzZWxmLm5hbWUgPSBuYW1lCiAgICAgICAgc2VsZi5yb2xlID0gcm9sZQogICAgICAgIHNlbGYuYmFja2VuZCA9IGJhY2tlbmQKICAgICAgICBzZWxmLmNhbGxzID0gMAogICAgZGVmIGFjdChzZWxmLCBtc2c6ICJNZXNzYWdlIikgLT4gIk1lc3NhZ2UiOgogICAgICAgIHNlbGYuY2FsbHMgKz0gMQogICAgICAgIG91dCwgdG9rZW5zID0gc2VsZi5iYWNrZW5kKHNlbGYubmFtZSwgbXNnLmNvbnRlbnQpCiAgICAgICAgcmV0dXJuIE1lc3NhZ2Uoc2VuZGVyPXNlbGYubmFtZSwgcmVjaXBpZW50PW1zZy5zZW5kZXIsCiAgICAgICAgICAgICAgICAgICAgICAga2luZD0icmVzdWx0IiwgY29udGVudD1vdXQsIG1ldGE9eyJ0b2tlbnMiOiB0b2tlbnN9KQo="
_ROUTER_B64 = "IyBvcmNoZXN0cmEvcm91dGVyLnB5ICAtLSAgYSByb3V0ZXIgdGhhdCBjbGFzc2lmaWVzIGVhY2ggdGFzayBhbmQgaGFuZHMgaXQKIyB0byB0aGUgYmVzdCBzcGVjaWFsaXN0LCBlc2NhbGF0ZXMgd2hlbiB1bnN1cmUsIGFuZCBzdXJ2aXZlcyBoYW5kb2ZmIGxvb3BzLgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsLCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgUm91dGU6CiAgICBjYXRlZ29yeTogT3B0aW9uYWxbc3RyXQogICAgY29uZmlkZW5jZTogZmxvYXQKCmNsYXNzIENsYXNzaWZpZXI6CiAgICAjIENoZWFwIGtleXdvcmQgY2xhc3NpZmllci4gSW4gcHJvZHVjdGlvbiB0aGlzIHdvdWxkIGJlIGFuIGVtYmVkZGluZyBtb2RlbAogICAgIyBvciBhIHNtYWxsIExMTTsgdGhlIGludGVyZmFjZSAodGV4dCAtPiBSb3V0ZSkgaXMgd2hhdCBtYXR0ZXJzLgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGtleXdvcmRzOiBEaWN0W3N0ciwgbGlzdF0pOgogICAgICAgIHNlbGYua2V5d29yZHMgPSBrZXl3b3JkcwogICAgICAgIHNlbGYuY2FsbHMgPSAwCiAgICBkZWYgY2xhc3NpZnkoc2VsZiwgdGV4dDogc3RyKSAtPiBSb3V0ZToKICAgICAgICBzZWxmLmNhbGxzICs9IDEKICAgICAgICB0ID0gdGV4dC5sb3dlcigpCiAgICAgICAgc2NvcmVzID0ge30KICAgICAgICBmb3IgY2F0LCBrd3MgaW4gc2VsZi5rZXl3b3Jkcy5pdGVtcygpOgogICAgICAgICAgICBoaXRzID0gc3VtKDEgZm9yIGsgaW4ga3dzIGlmIGsgaW4gdCkKICAgICAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgICAgIHNjb3Jlc1tjYXRdID0gaGl0cwogICAgICAgIGlmIG5vdCBzY29yZXM6CiAgICAgICAgICAgIHJldHVybiBSb3V0ZShOb25lLCAwLjApICAgICAgICAgICAgICAjIG5vdGhpbmcgbWF0Y2hlZCAtPiBlc2NhbGF0ZQogICAgICAgIHRvdGFsID0gc3VtKHNjb3Jlcy52YWx1ZXMoKSkKICAgICAgICBiZXN0X2NhdCwgYmVzdF9oaXRzID0gc29ydGVkKHNjb3Jlcy5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoLWt2WzFdLCBrdlswXSkpWzBdCiAgICAgICAgcmV0dXJuIFJvdXRlKGJlc3RfY2F0LCBiZXN0X2hpdHMgLyB0b3RhbCkKCmNsYXNzIFJvdXRlcjoKICAgICMgSG9sZHMgc3BlY2lhbGlzdHMgKGNhdGVnb3J5IC0+IEFnZW50KSwgYSBnZW5lcmFsaXN0IGZhbGxiYWNrLCBhbmQgdGhlCiAgICAjIHBvbGljeTogZ2F0ZSBvbiBjb25maWRlbmNlLCBkaXNwYXRjaCwgZm9sbG93IGhhbmRvZmZzLCBjYXAgdGhlIGhvcHMuCiAgICBkZWYgX19pbml0X18oc2VsZiwgY2xhc3NpZmllciwgc3BlY2lhbGlzdHMsIGdlbmVyYWxpc3QsCiAgICAgICAgICAgICAgICAgdGhyZXNob2xkPTAuNiwgaG9wX2NhcD0zLCBjbGFzc2lmeV9jb3N0PTIpOgogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IGNsYXNzaWZpZXIKICAgICAgICBzZWxmLnNwZWNpYWxpc3RzID0gZGljdChzcGVjaWFsaXN0cykKICAgICAgICBzZWxmLmdlbmVyYWxpc3QgPSBnZW5lcmFsaXN0CiAgICAgICAgc2VsZi50aHJlc2hvbGQgPSB0aHJlc2hvbGQKICAgICAgICBzZWxmLmhvcF9jYXAgPSBob3BfY2FwCiAgICAgICAgc2VsZi5jbGFzc2lmeV9jb3N0ID0gY2xhc3NpZnlfY29zdAoKICAgIGRlZiBfZXNjYWxhdGUoc2VsZiwgdGFzaywgdHJhY2UsIHRva2VucywgcmVhc29uKToKICAgICAgICBob3BzID0gbGVuKHRyYWNlKQogICAgICAgIG0gPSBzZWxmLmdlbmVyYWxpc3QuYWN0KE1lc3NhZ2UoInJvdXRlciIsIHNlbGYuZ2VuZXJhbGlzdC5uYW1lLCAidGFzayIsIHRhc2spKQogICAgICAgIHRyYWNlLmFwcGVuZChzZWxmLmdlbmVyYWxpc3QubmFtZSkKICAgICAgICB0b2tlbnMgKz0gbS5tZXRhLmdldCgidG9rZW5zIiwgMCkKICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywgInRyYWNlIjogdHJhY2UsCiAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6IHJlYXNvbiwgImhvcHMiOiBob3BzfQoKICAgIGRlZiBkaXNwYXRjaChzZWxmLCB0YXNrKToKICAgICAgICB0cmFjZSA9IFtdCiAgICAgICAgdG9rZW5zID0gc2VsZi5jbGFzc2lmeV9jb3N0CiAgICAgICAgcm91dGUgPSBzZWxmLmNsYXNzaWZpZXIuY2xhc3NpZnkodGFza1sidGV4dCJdKQogICAgICAgIGlmIHJvdXRlLmNhdGVnb3J5IGlzIE5vbmUgb3Igcm91dGUuY29uZmlkZW5jZSA8IHNlbGYudGhyZXNob2xkOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgImxvd19jb25maWRlbmNlIikKICAgICAgICBjdXJyZW50ID0gcm91dGUuY2F0ZWdvcnkKICAgICAgICBob3BzID0gMAogICAgICAgIHdoaWxlIGhvcHMgPCBzZWxmLmhvcF9jYXA6CiAgICAgICAgICAgIGhvcHMgKz0gMQogICAgICAgICAgICBpZiBjdXJyZW50IG5vdCBpbiBzZWxmLnNwZWNpYWxpc3RzOgogICAgICAgICAgICAgICAgcmV0dXJuIHNlbGYuX2VzY2FsYXRlKHRhc2ssIHRyYWNlLCB0b2tlbnMsICJ1bmtub3duX2NhdGVnb3J5IikKICAgICAgICAgICAgYWdlbnQgPSBzZWxmLnNwZWNpYWxpc3RzW2N1cnJlbnRdCiAgICAgICAgICAgIG0gPSBhZ2VudC5hY3QoTWVzc2FnZSgicm91dGVyIiwgYWdlbnQubmFtZSwgInRhc2siLCB0YXNrKSkKICAgICAgICAgICAgdHJhY2UuYXBwZW5kKGFnZW50Lm5hbWUpCiAgICAgICAgICAgIHRva2VucyArPSBtLm1ldGEuZ2V0KCJ0b2tlbnMiLCAwKQogICAgICAgICAgICB0YXJnZXQgPSBtLmNvbnRlbnQuZ2V0KCJoYW5kb2ZmIikKICAgICAgICAgICAgaWYgdGFyZ2V0IGlzIE5vbmU6CiAgICAgICAgICAgICAgICByZXR1cm4geyJhbnN3ZXIiOiBtLmNvbnRlbnQuZ2V0KCJhbnN3ZXIiKSwgInRva2VucyI6IHRva2VucywKICAgICAgICAgICAgICAgICAgICAgICAgInRyYWNlIjogdHJhY2UsICJlc2NhbGF0ZWQiOiBGYWxzZSwgInJlYXNvbiI6ICJyb3V0ZWQiLCAiaG9wcyI6IGhvcHN9CiAgICAgICAgICAgIGN1cnJlbnQgPSB0YXJnZXQKICAgICAgICByZXR1cm4gc2VsZi5fZXNjYWxhdGUodGFzaywgdHJhY2UsIHRva2VucywgImhvcF9jYXAiKQo="
_PLANNER_B64= "IyBvcmNoZXN0cmEvcGxhbm5lci5weSAgLS0gIGRlY29tcG9zZSBhIGdvYWwgaW50byBhbiBvcmRlcmVkIHBsYW4gKGEgREFHIG9mCiMgc3ViLXRhc2tzKSwgZGlzcGF0Y2ggZWFjaCBzdWItdGFzayB0aHJvdWdoIHRoZSBMZXNzb24tNzkgUm91dGVyLCB0aHJlYWQgZWFjaAojIGRlcGVuZGVuY3kncyBvdXRwdXQgaW50byB0aGUgdGFza3MgdGhhdCBkZXBlbmQgb24gaXQsIGFuZCBhc3NlbWJsZSBhIHJlc3VsdC4KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBmaWVsZApmcm9tIHR5cGluZyBpbXBvcnQgQ2FsbGFibGUsIE9wdGlvbmFsLCBMaXN0LCBEaWN0CmZyb20gb3JjaGVzdHJhLmNvcmUgaW1wb3J0IE1lc3NhZ2UKCkBkYXRhY2xhc3MKY2xhc3MgU3RlcDoKICAgIGlkOiBzdHIKICAgIHRleHQ6IHN0cgogICAgZGVwczogTGlzdFtzdHJdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWxpc3QpICAgIyBpZHMgdGhpcyBzdGVwIHdhaXRzIG9uCiAgICBwYXlsb2FkOiBkaWN0ID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpICAgICAgIyB0aGUgYXRvbWljIHRhc2sgZmllbGRzCgpAZGF0YWNsYXNzCmNsYXNzIFBsYW46CiAgICBnb2FsOiBzdHIKICAgIHN0ZXBzOiBMaXN0W1N0ZXBdCiAgICBkZWYgYnlfaWQoc2VsZikgLT4gRGljdFtzdHIsICJTdGVwIl06CiAgICAgICAgcmV0dXJuIHtzLmlkOiBzIGZvciBzIGluIHNlbGYuc3RlcHN9CgpkZWYgdG9wb19vcmRlcihzdGVwczogTGlzdFtTdGVwXSkgLT4gTGlzdFtzdHJdOgogICAgIyBLYWhuJ3MgYWxnb3JpdGhtLiBSYWlzZXMgVmFsdWVFcnJvciBvbiBhIG1pc3NpbmcgZGVwIG9yIGEgY3ljbGUuCiAgICBpZHMgPSB7cy5pZCBmb3IgcyBpbiBzdGVwc30KICAgIGluZGVnID0ge3MuaWQ6IDAgZm9yIHMgaW4gc3RlcHN9CiAgICBhZGogPSB7cy5pZDogW10gZm9yIHMgaW4gc3RlcHN9CiAgICBmb3IgcyBpbiBzdGVwczoKICAgICAgICBmb3IgZCBpbiBzLmRlcHM6CiAgICAgICAgICAgIGlmIGQgbm90IGluIGlkczoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInN0ZXAgJXIgZGVwZW5kcyBvbiB1bmtub3duIHN0ZXAgJXIiICUgKHMuaWQsIGQpKQogICAgICAgICAgICBhZGpbZF0uYXBwZW5kKHMuaWQpCiAgICAgICAgICAgIGluZGVnW3MuaWRdICs9IDEKICAgIHJlYWR5ID0gc29ydGVkKFtpIGZvciBpIGluIGluZGVnIGlmIGluZGVnW2ldID09IDBdKSAgICMgZGV0ZXJtaW5pc3RpYyBvcmRlcgogICAgb3JkZXIgPSBbXQogICAgd2hpbGUgcmVhZHk6CiAgICAgICAgbiA9IHJlYWR5LnBvcCgwKQogICAgICAgIG9yZGVyLmFwcGVuZChuKQogICAgICAgIGZvciBtIGluIGFkaltuXToKICAgICAgICAgICAgaW5kZWdbbV0gLT0gMQogICAgICAgICAgICBpZiBpbmRlZ1ttXSA9PSAwOgogICAgICAgICAgICAgICAgcmVhZHkuYXBwZW5kKG0pCiAgICAgICAgcmVhZHkuc29ydCgpCiAgICBpZiBsZW4ob3JkZXIpICE9IGxlbihzdGVwcyk6CiAgICAgICAgc3R1Y2sgPSBsZW4oc3RlcHMpIC0gbGVuKG9yZGVyKQogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBsYW4gaGFzIGEgY3ljbGU7ICVkIG9mICVkIHN0ZXBzIG5ldmVyIGJlY2FtZSByZWFkeSIKICAgICAgICAgICAgICAgICAgICAgICAgICUgKHN0dWNrLCBsZW4oc3RlcHMpKSkKICAgIHJldHVybiBvcmRlcgoKZGVmIGxheWVycyhzdGVwczogTGlzdFtTdGVwXSkgLT4gTGlzdFtMaXN0W3N0cl1dOgogICAgIyBHcm91cCBzdGVwcyBpbnRvIGRlcGVuZGVuY3kgbGF5ZXJzLiBFdmVyeSBzdGVwIGluIGEgbGF5ZXIgY2FuIHJ1biBpbgogICAgIyBwYXJhbGxlbDsgbGF5ZXIgayBkZXBlbmRzIG9ubHkgb24gbGF5ZXJzIDwgay4gVmFsaWRhdGVzIChyYWlzZXMgb24gY3ljbGUpLgogICAgb3JkZXIgPSB0b3BvX29yZGVyKHN0ZXBzKQogICAgYnlfaWQgPSB7cy5pZDogcyBmb3IgcyBpbiBzdGVwc30KICAgIGRlcHRoID0ge30KICAgIGZvciBzaWQgaW4gb3JkZXI6CiAgICAgICAgZHMgPSBieV9pZFtzaWRdLmRlcHMKICAgICAgICBkZXB0aFtzaWRdID0gMCBpZiBub3QgZHMgZWxzZSAxICsgbWF4KGRlcHRoW2RdIGZvciBkIGluIGRzKQogICAgb3V0ID0gW10KICAgIGZvciBzaWQgaW4gb3JkZXI6CiAgICAgICAgZCA9IGRlcHRoW3NpZF0KICAgICAgICB3aGlsZSBsZW4ob3V0KSA8PSBkOgogICAgICAgICAgICBvdXQuYXBwZW5kKFtdKQogICAgICAgIG91dFtkXS5hcHBlbmQoc2lkKQogICAgcmV0dXJuIFtzb3J0ZWQobCkgZm9yIGwgaW4gb3V0XQoKY2xhc3MgUGxhbm5lcjoKICAgICMgZGVjb21wb3NlIC0+IG9yZGVyIC0+IGRpc3BhdGNoIGVhY2ggdmlhIHRoZSBSb3V0ZXIgKHRocmVhZGluZyBkZXBlbmRlbmN5CiAgICAjIG91dHB1dHMgZm9yd2FyZCkgLT4gdmFsaWRhdGUgZWFjaCByZXN1bHQgYW5kIFJFUExBTiBmYWlsdXJlcyBvbiB0aGUKICAgICMgZ2VuZXJhbGlzdCAtPiBhc3NlbWJsZS4gVGhlIFJvdXRlciBhbnN3ZXJzICJ3aG8iOyB0aGUgUGxhbm5lciBhbnN3ZXJzCiAgICAjICJ3aGF0IGFyZSB0aGUgdGFza3MsIGluIHdoYXQgb3JkZXIsIGFuZCBkaWQgZWFjaCBvbmUgYWN0dWFsbHkgc3VjY2VlZCIuCiAgICBkZWYgX19pbml0X18oc2VsZiwgZGVjb21wb3NlOiBDYWxsYWJsZSwgcm91dGVyLAogICAgICAgICAgICAgICAgIHZhbGlkYXRlOiBPcHRpb25hbFtDYWxsYWJsZV0gPSBOb25lKToKICAgICAgICBzZWxmLmRlY29tcG9zZSA9IGRlY29tcG9zZSAgICAgICAgICAgICAgICMgZ29hbF90ZXh0IC0+IFBsYW4KICAgICAgICBzZWxmLnJvdXRlciA9IHJvdXRlcgogICAgICAgIHNlbGYudmFsaWRhdGUgPSB2YWxpZGF0ZSBvciAobGFtYmRhIHN0ZXAsIG91dDogb3V0LmdldCgiYW5zd2VyIikgaXMgbm90IE5vbmUpCgogICAgZGVmIHBsYW4oc2VsZiwgZ29hbF90ZXh0OiBzdHIpIC0+IFBsYW46CiAgICAgICAgcmV0dXJuIHNlbGYuZGVjb21wb3NlKGdvYWxfdGV4dCkKCiAgICBkZWYgZXhlY3V0ZShzZWxmLCBwbGFuOiBQbGFuKSAtPiBkaWN0OgogICAgICAgIG9yZGVyID0gdG9wb19vcmRlcihwbGFuLnN0ZXBzKSAgICAgICAgICAgIyByYWlzZXMgb24gYSBiYWQgcGxhbgogICAgICAgIGJ5X2lkID0gcGxhbi5ieV9pZCgpCiAgICAgICAgcmVzdWx0cywgcmVwbGFubmVkID0ge30sIFtdCiAgICAgICAgZm9yIHNpZCBpbiBvcmRlcjoKICAgICAgICAgICAgc3RlcCA9IGJ5X2lkW3NpZF0KICAgICAgICAgICAgdGFzayA9IGRpY3Qoc3RlcC5wYXlsb2FkKQogICAgICAgICAgICB0YXNrWyJ0ZXh0Il0gPSBzdGVwLnRleHQKICAgICAgICAgICAgIyBUSFJFQUQgZWFjaCBkZXBlbmRlbmN5J3MgYW5zd2VyIGludG8gdGhpcyB0YXNrJ3MgY29udGV4dC4KICAgICAgICAgICAgdGFza1siY29udGV4dCJdID0ge2Q6IHJlc3VsdHNbZF0uZ2V0KCJhbnN3ZXIiKSBmb3IgZCBpbiBzdGVwLmRlcHN9CiAgICAgICAgICAgIG91dCA9IHNlbGYucm91dGVyLmRpc3BhdGNoKHRhc2spCiAgICAgICAgICAgIGlmIG5vdCBzZWxmLnZhbGlkYXRlKHN0ZXAsIG91dCk6CiAgICAgICAgICAgICAgICAjIFJFUExBTjogd2hvZXZlciB0aGUgcm91dGVyIHBpY2tlZCBmYWlsZWQuIEVzY2FsYXRlIHRoaXMgb25lCiAgICAgICAgICAgICAgICAjIHN0ZXAgc3RyYWlnaHQgdG8gdGhlIGdlbmVyYWxpc3QgKGl0IGtlZXBzIHRoZSB0aHJlYWRlZCBjb250ZXh0KS4KICAgICAgICAgICAgICAgIGcgPSBzZWxmLnJvdXRlci5nZW5lcmFsaXN0CiAgICAgICAgICAgICAgICBtID0gZy5hY3QoTWVzc2FnZSgicGxhbm5lciIsIGcubmFtZSwgInRhc2siLCB0YXNrKSkKICAgICAgICAgICAgICAgIG91dCA9IHsiYW5zd2VyIjogbS5jb250ZW50LmdldCgiYW5zd2VyIiksCiAgICAgICAgICAgICAgICAgICAgICAgInRva2VucyI6IG91dC5nZXQoInRva2VucyIsIDApICsgbS5tZXRhLmdldCgidG9rZW5zIiwgMCksCiAgICAgICAgICAgICAgICAgICAgICAgInRyYWNlIjogb3V0LmdldCgidHJhY2UiLCBbXSkgKyBbZy5uYW1lXSwKICAgICAgICAgICAgICAgICAgICAgICAiZXNjYWxhdGVkIjogVHJ1ZSwgInJlYXNvbiI6ICJyZXBsYW4ifQogICAgICAgICAgICAgICAgcmVwbGFubmVkLmFwcGVuZChzaWQpCiAgICAgICAgICAgIHJlc3VsdHNbc2lkXSA9IG91dAogICAgICAgIHJldHVybiB7ImdvYWwiOiBwbGFuLmdvYWwsICJvcmRlciI6IG9yZGVyLCAicmVzdWx0cyI6IHJlc3VsdHMsCiAgICAgICAgICAgICAgICAicmVwbGFubmVkIjogcmVwbGFubmVkLAogICAgICAgICAgICAgICAgImFuc3dlcnMiOiB7azogdi5nZXQoImFuc3dlciIpIGZvciBrLCB2IGluIHJlc3VsdHMuaXRlbXMoKX0sCiAgICAgICAgICAgICAgICAidG9rZW5zIjogc3VtKHYuZ2V0KCJ0b2tlbnMiLCAwKSBmb3IgdiBpbiByZXN1bHRzLnZhbHVlcygpKX0K"
for fname, b in [("core.py", _CORE_B64), ("router.py", _ROUTER_B64), ("planner.py", _PLANNER_B64)]:
    with open(os.path.join(BASE, "orchestra", fname), "w") as f:
        f.write(base64.b64decode(b).decode())
open(os.path.join(BASE, "orchestra", "__init__.py"), "w").close()

if BASE not in sys.path:
    sys.path.insert(0, BASE)
importlib.invalidate_caches()

from orchestra.core import Message, Agent
from orchestra.router import Classifier, Route, Router
from orchestra.planner import Step, Plan, topo_order, layers, Planner
print("orchestra package ready:", os.listdir(os.path.join(BASE, "orchestra")))

In [ ]:
# --- The deterministic world: keywords, a solver, specialists, a generalist ---
import re

# Each clause of a goal uses keywords from exactly ONE specialty, so a single
# clause routes at confidence 1.0. A whole multi-clause goal mixes specialties.
KEYWORDS = {
    "translate": ["translate", "french", "memo", "language"],
    "math":      ["compute", "total", "tax", "subtotal", "sum"],
    "billing":   ["bill", "customer", "invoice", "charge"],
    "code":      ["write", "function", "script", "refactor"],
}
clf = Classifier(KEYWORDS)

FR2EN = {"la facture": "the invoice", "le recu": "the receipt", "le devis": "the quote"}

def solve_atomic(task):
    # The ground-truth solver for ONE atomic task. Returns an answer dict, or
    # None if a required input is missing (e.g. billing with no amount yet).
    c   = task.get("category")
    ctx = task.get("context", {}) or {}
    if c == "translate":
        return {"text_en": FR2EN.get(task.get("fr", ""), "?")}
    if c == "math":
        return {"total": round(task["subtotal"] * (1 + task["tax"]), 2)}
    if c == "code":
        return {"code": "def %s(): ..." % task.get("what", "f")}
    if c == "billing":
        amt = None
        for v in ctx.values():                     # pull an upstream computed total
            if isinstance(v, dict) and "total" in v:
                amt = v["total"]
        if amt is None:
            amt = task.get("amount")               # or an amount handed in directly
        if amt is None:
            return None                            # <- cannot bill without an amount
        return {"invoice": "Billed %s $%.2f" % (task.get("customer", "customer"), amt),
                "amount": amt}
    return None

def make_specialist(cat):
    # A specialist solves ONLY its own category and is blind to everything else.
    # If handed raw text with no structured fields (e.g. a whole compound goal
    # routed to it), it parses ITS OWN slice out and solves just that.
    def backend(name, task):
        t = dict(task)
        if "category" not in t:
            try:
                t.update(extract_payload(cat, t.get("text", "")))
            except Exception:
                t["category"] = cat
        if t.get("category") != cat:
            return {"answer": None, "handoff": None}, 20       # off-domain -> blank
        try:
            ans = solve_atomic(t)
        except Exception:
            ans = None                                         # missing params -> blank
        return {"answer": ans, "handoff": None}, 20            # ans may be None if input missing
    return backend

def generalist_backend(name, task):
    # Competent on any SINGLE atomic task, but 3x the cost. Given a whole
    # multi-clause goal (no category set) it can only finish the FIRST clause
    # and stops -- exactly why routing-alone drops the rest of a compound goal.
    if task.get("category"):
        return {"answer": solve_atomic(task), "handoff": None}, 60
    first = decompose(task.get("text", "")).steps[0]
    sub = dict(first.payload); sub["text"] = first.text; sub["context"] = {}
    return {"answer": solve_atomic(sub), "handoff": None}, 60

SPECIALTIES = ["translate", "math", "billing", "code"]
specialists = {c: Agent(c + "-bot", c, make_specialist(c)) for c in SPECIALTIES}
generalist  = Agent("generalist", "generalist", generalist_backend)
router = Router(clf, specialists, generalist, threshold=0.6)
print("specialists:", list(specialists), "| generalist:", generalist.name)

In [ ]:
# --- decompose(): goal text  ->  Plan (a DAG of Steps) ---
# " then "     = the next clause DEPENDS on the previous one (ordered).
# " and also " = the next clause is INDEPENDENT (can run in parallel).
# In production this is one LLM call; the interface (text -> Plan) is the point.
def extract_payload(cat, clause):
    if cat == "translate":
        m = re.search(r"'([^']+)'", clause);            return {"category": "translate", "fr": m.group(1)}
    if cat == "math":
        s = int(re.search(r"subtotal (\d+)", clause).group(1))
        t = float(re.search(r"tax ([\d.]+)", clause).group(1))
        return {"category": "math", "subtotal": s, "tax": t}
    if cat == "billing":
        m = re.search(r"customer (\w+)", clause);       return {"category": "billing", "customer": m.group(1)}
    if cat == "code":
        m = re.search(r"a (\w+) function", clause);      return {"category": "code", "what": m.group(1)}
    return {"category": cat}

def decompose(goal):
    parts   = re.split(r"(\s+then\s+|\s+and also\s+)", goal)
    clauses = [parts[0]]
    conns   = [None]
    for i in range(1, len(parts), 2):
        conns.append(parts[i].strip())
        clauses.append(parts[i + 1])
    steps = []
    for idx, cl in enumerate(clauses):
        cat  = clf.classify(cl).category
        deps = [steps[idx - 1].id] if idx > 0 and conns[idx] == "then" else []
        steps.append(Step(id="s%d" % idx, text=cl.strip(),
                          deps=deps, payload=extract_payload(cat, cl)))
    return Plan(goal=goal, steps=steps)

demo = decompose("compute the total for subtotal 100 at tax 0.1 then bill customer acme")
for s in demo.steps:
    print(s.id, "|", s.payload["category"], "| deps:", s.deps, "|", s.text)

## 1. The problem — a router can only pick ONE specialist

Watch a router handle a compound goal. It classifies the **whole sentence**,
picks the single **loudest** specialist (or escalates to the generalist), gets
**one** answer back — the dominant slice — and the rest of the goal is simply
gone.

In [ ]:
GOAL = "compute the total for subtotal 100 at tax 0.1 then bill customer acme"

# Route the WHOLE goal as if it were one task.
r = router.dispatch({"text": GOAL})
print("router-alone answer :", r["answer"])
print("router-alone trace  :", r["trace"], "| escalated:", r["escalated"])

# The goal actually needs TWO sub-answers: a computed total AND a bill.
plan = decompose(GOAL)
need = [s.payload["category"] for s in plan.steps]
print("\\nthe goal really needs:", need, "->", len(need), "sub-tasks")
print("router-alone produced :", 1, "answer  ->  the bill is missing entirely")

assert r["answer"] is not None                 # it did solve *something*
assert "invoice" not in (r["answer"] or {})    # ...but NOT the billing step
print("\\nRouting alone is structurally incomplete on a compound goal. We need a plan.")

## 2. The plan as a DAG — order is not optional

A plan is a **directed acyclic graph**: nodes are sub-tasks, edges are
"must-finish-before". Two jobs fall out of that graph:

- **`topo_order`** — a valid linear order to run the steps (Kahn's algorithm).
  Give it a **cycle** and it must *refuse*, not loop forever.
- **`layers`** — steps grouped so everything in a layer is independent and can
  run **in parallel**; layer *k* only waits on the layers before it.

Note the payoff below: even when the steps are handed in **backwards**, the
topological order still runs the dependency first.

In [ ]:
# (a) Out-of-order input still yields dependency-correct order.
scrambled = Plan("x", [
    Step("bill", "bill customer acme",       deps=["calc"], payload={"category": "billing", "customer": "acme"}),
    Step("calc", "compute the total ...",    deps=[],        payload={"category": "math", "subtotal": 100, "tax": 0.1}),
])
order = topo_order(scrambled.steps)
print("input order :", [s.id for s in scrambled.steps])
print("run order   :", order)
assert order == ["calc", "bill"], "the dependency must run first"

# (b) A cycle is rejected -- not run, not looped.
cyclic = [Step("a", "a", deps=["b"]), Step("b", "b", deps=["a"])]
try:
    topo_order(cyclic)
    raise SystemExit("should have raised")
except ValueError as e:
    print("cycle correctly rejected ->", e)

# (c) Layers expose parallelism. Three independent producers -> one layer of 3.
par = [Step("p", "translate ... 'la facture'", payload={"category": "translate", "fr": "la facture"}),
       Step("q", "compute ... subtotal 100 tax 0.1", payload={"category": "math", "subtotal": 100, "tax": 0.1}),
       Step("r", "write a sort function",       payload={"category": "code", "what": "sort"})]
print("\\nlayers(3 independent):", layers(par))
assert layers(par) == [["p", "q", "r"]]
assert layers(scrambled.steps) == [["calc"], ["bill"]]   # a real 2-deep chain
print("layers(calc->bill)   :", layers(scrambled.steps))

## 3. The Planner — decompose → order → dispatch → thread → assemble  ← THE PAYOFF

Now the whole loop. The `Planner` decomposes the goal, orders the steps, and
for each one **builds a task, threads in every dependency's answer as
`context`, and dispatches it through the Router**. The billing step can't
invent an amount — it *reads the total the math step produced*. That thread is
the difference between a list of tasks and a **plan**.

In [ ]:
planner = Planner(decompose, router)

plan   = planner.plan(GOAL)
result = planner.execute(plan)

print("run order :", result["order"])
for sid, ans in result["answers"].items():
    print(" ", sid, "->", ans)

calc = result["answers"]["s0"]     # the math step
bill = result["answers"]["s1"]     # the billing step (depended on s0)

# THE PAYOFF: both sub-tasks solved, AND the bill used the computed total.
assert calc == {"total": 110.0}
assert bill["amount"] == 110.0 and bill["invoice"] == "Billed acme $110.00"
print("\\nBoth sub-tasks solved. The bill ($110.00) = the total the math step computed.")
print("The dependency was threaded, not guessed. That is planning.")

## 4. Measured: planner vs router-alone over a suite of goals

One example is an anecdote. Let's score **completeness** — *what fraction of a
goal's sub-tasks got a correct answer* — over 40 generated compound goals.

- **Router-alone** gets one dispatch per goal → at best it finishes the first
  clause → completeness ≈ `1 / (number of steps)`.
- **Planner** decomposes and runs every step → completeness `1.0`.

In [ ]:
import random

def gen_goal(seed):
    rng = random.Random(seed)
    tmpl = rng.choice(["math_bill", "translate_code", "math_bill_translate", "translate_math_bill"])
    s, t = rng.choice([80, 100, 150, 200]), rng.choice([0.0, 0.1, 0.2])
    cust = rng.choice(["acme", "globex", "initech"]);  fr = rng.choice(list(FR2EN))
    w = rng.choice(["sort", "parse", "hash"])
    C_MATH = "compute the total for subtotal %d at tax %s" % (s, t)
    C_BILL = "bill customer %s" % cust
    C_TRAN = "translate the french memo '%s'" % fr
    C_CODE = "write a %s function" % w
    if tmpl == "math_bill":            return C_MATH + " then " + C_BILL
    if tmpl == "translate_code":       return C_TRAN + " and also " + C_CODE
    if tmpl == "math_bill_translate":  return C_MATH + " then " + C_BILL + " and also " + C_TRAN
    return C_TRAN + " and also " + C_MATH + " then " + C_BILL

def completeness(goal):
    plan   = decompose(goal)
    truth  = {s.id: solve_ground_truth(plan, s) for s in plan.steps}
    k      = len(plan.steps)
    # planner: run everything
    got    = planner.execute(plan)["answers"]
    p_ok   = sum(1 for sid in truth if got.get(sid) == truth[sid]) / k
    # router-alone: one dispatch for the whole goal
    ra     = router.dispatch({"text": goal})["answer"]
    r_ok   = sum(1 for sid in truth if truth[sid] == ra) / k
    return p_ok, r_ok

def solve_ground_truth(plan, step):
    # compute the correct answer for a step, honoring its dependency threading
    by = plan.by_id(); memo = {}
    def rec(sid):
        if sid in memo: return memo[sid]
        s = by[sid]
        task = dict(s.payload); task["context"] = {d: rec(d) for d in s.deps}
        memo[sid] = solve_atomic(task); return memo[sid]
    return rec(step.id)

pairs = [completeness(gen_goal(i)) for i in range(40)]
planner_acc = sum(p for p, _ in pairs) / len(pairs)
router_acc  = sum(r for _, r in pairs) / len(pairs)
print("planner completeness    : %.2f" % planner_acc)
print("router-alone completeness: %.2f" % router_acc)
assert planner_acc >= 0.99
assert planner_acc > router_acc + 0.5
print("\\nSame router, same specialists. The plan is what makes the goal finish.")

## 5. Parallelism for free — the plan already knows what's independent

Because the plan is a DAG, `layers()` tells you exactly which steps can run at
the same time. Independent producers ("… **and also** …") land in one layer and
fan out. Here we run each layer concurrently with a thread pool and measure it.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

def run_step(step, results):
    task = dict(step.payload); task["text"] = step.text
    task["context"] = {d: results[d] for d in step.deps}
    time.sleep(0.06)                       # pretend each specialist call takes 60ms
    return step.id, solve_atomic(task)

goal = "translate the french memo 'la facture' and also compute the total for subtotal 100 at tax 0.1 and also write a sort function"
steps = decompose(goal).steps
lys   = layers(steps); by = {s.id: s for s in steps}
print("layers:", lys)

# sequential
res = {}; t0 = time.time()
for L in lys:
    for sid in L:
        _, a = run_step(by[sid], res); res[sid] = a
seq_t = time.time() - t0

# parallel within each layer
res2 = {}; t0 = time.time()
for L in lys:
    with ThreadPoolExecutor(max_workers=len(L)) as ex:
        for sid, a in ex.map(lambda s: run_step(by[s], res2), L):
            res2[sid] = a
par_t = time.time() - t0

print("sequential: %.2fs | parallel: %.2fs | speed-up: %.1fx" % (seq_t, par_t, seq_t / par_t))
assert res == res2                         # identical results
assert par_t < seq_t * 0.7                 # 3 independent steps -> ~1 layer of work
print("Same answers, a fraction of the wall-clock. The DAG gave us the parallelism.")

## 6. When a step fails — replanning

Routing decides *who*. But the chosen specialist can still **fail** (bad input,
a transient error, a wrong route that got through the gate). A plan is only
robust if it **checks each result and recovers**. The `Planner` validates every
step; a failure is **replanned** onto the generalist — which keeps the threaded
context, so it can still finish the job.

In [ ]:
# A billing specialist that is currently broken (always returns a blank answer).
def broken_billing(name, task):
    return {"answer": None, "handoff": None}, 20

router_flaky = Router(clf, {**specialists, "billing": Agent("billing-bot", "billing", broken_billing)},
                      generalist, threshold=0.6)
planner_flaky = Planner(decompose, router_flaky)

res = planner_flaky.execute(decompose(GOAL))   # math -> bill
print("replanned steps:", res["replanned"])
print("billing answer :", res["answers"]["s1"])

assert res["replanned"] == ["s1"]                          # the billing step was rescued
assert res["answers"]["s1"]["invoice"] == "Billed acme $110.00"   # ...and still correct
assert res["results"]["s1"]["reason"] == "replan"
# it cost more (generalist = 60 tok) but the goal still completed
print("tokens (with a replan):", res["tokens"], "-- a rescue costs more, but the goal finishes")

## 7. Ten ways planning goes wrong

| # | Pitfall | Symptom | Fix |
|---|---|---|---|
| 1 | **Planning a trivial goal** | one atomic task wrapped in DAG overhead | plan only when the goal is compound; route the rest |
| 2 | **Missing a step** | a needed sub-task never appears | validate the plan against required outputs before running |
| 3 | **Wrong dependency** | step runs before its input exists | make data deps explicit; thread outputs as `context` |
| 4 | **A cycle** | executor loops or hangs | `topo_order` must raise on cycles (it does) |
| 5 | **Over-decomposition** | 20 tiny steps, 20× the cost | coarser steps; merge trivially-chained clauses |
| 6 | **No result validation** | a blank step silently poisons downstream | validate every step; replan failures |
| 7 | **No replanning** | one failure sinks the whole goal | escalate/retry the failed step, keep its context |
| 8 | **Lost context across steps** | downstream re-derives or guesses | pass upstream answers forward explicitly |
| 9 | **Serial-only execution** | independent steps wait needlessly | use `layers()` to parallelize |
| 10 | **No plan logging** | can't tell *why* a goal failed | emit one trace span per step (wire L72 tracing) |

## 8. Ship it & verify — `orchestra/planner.py` is a real module

`planner.py` is already on disk and imported. Re-import it fresh, smoke-test the
public surface, then re-assert every claim this lesson made.

In [ ]:
import importlib, orchestra.planner as P
importlib.reload(P)

# smoke test 1: topo_order respects deps and is deterministic
s = [P.Step("b", "b", deps=["a"]), P.Step("a", "a")]
assert P.topo_order(s) == ["a", "b"]

# smoke test 2: layers group independent work
s2 = [P.Step("x", "x"), P.Step("y", "y"), P.Step("z", "z", deps=["x"])]
assert P.layers(s2) == [["x", "y"], ["z"]]

# smoke test 3: a fresh Planner solves the canonical goal end-to-end
pl = P.Planner(decompose, router)
out = pl.execute(pl.plan(GOAL))
assert out["answers"]["s1"]["amount"] == 110.0
print("orchestra.planner smoke tests passed:", ["topo_order", "layers", "Planner.execute"])

In [ ]:
# --- Verification checklist: every claim this lesson made, re-asserted ---
checks = []
def check(name, cond):
    checks.append((name, bool(cond)))
    print(("PASS " if cond else "FAIL ") + name)

# routing alone is incomplete on a compound goal
ra = router.dispatch({"text": GOAL})["answer"]
check("router-alone solves <= 1 sub-task",      "invoice" not in (ra or {}))

# the DAG machinery
check("topo runs dependency first",              topo_order(scrambled.steps) == ["calc", "bill"])
check("layers expose parallelism",               layers(par) == [["p", "q", "r"]])
_cycle_ok = False
try: topo_order([Step("a","a",deps=["b"]), Step("b","b",deps=["a"])])
except ValueError: _cycle_ok = True
check("cycle is rejected",                        _cycle_ok)
_missing_ok = False
try: topo_order([Step("a","a",deps=["ghost"])])
except ValueError: _missing_ok = True
check("missing dependency is rejected",           _missing_ok)

# the planner end-to-end + threading
res = planner.execute(decompose(GOAL))
check("planner solves every sub-task",            all(v is not None for v in res["answers"].values()))
check("dependency threaded (bill == total)",      res["answers"]["s1"]["amount"] == 110.0)

# measured superiority
check("planner completeness >= 0.99",             planner_acc >= 0.99)
check("planner beats router-alone by > 0.5",      planner_acc > router_acc + 0.5)

# replanning
_rf = Planner(decompose, Router(clf, {**specialists, "billing": Agent("billing-bot","billing",broken_billing)}, generalist, threshold=0.6))
_rr = _rf.execute(decompose(GOAL))
check("failed step is replanned",                 _rr["replanned"] == ["s1"])
check("replanned step still correct",             _rr["answers"]["s1"]["amount"] == 110.0)

passed = sum(1 for _, ok in checks if ok)
print("\\n%d / %d checks passed" % (passed, len(checks)))
assert passed == len(checks)

## 9. Summary, homework & what's next

### What you built
| Concept | What it does |
|---|---|
| **Decomposition** | one goal → many atomic sub-tasks (`decompose`) |
| **Plan / DAG** | sub-tasks + dependency edges (`Step`, `Plan`) |
| **`topo_order`** | a valid run order; refuses cycles & missing deps |
| **`layers`** | groups independent steps for parallel execution |
| **Context threading** | each result feeds the tasks that depend on it |
| **Replanning** | validate every step; rescue failures on the generalist |
| **Planner** | the conductor tying decompose → order → dispatch → assemble |

The through-line: **the Router answers *who*; the Planner answers *what, in what
order, and did each step actually succeed*.** Routing decides one task up front;
planning turns a goal into a graph of tasks and drives it to completion.

### Homework
1. **LLM decomposer.** Replace the keyword `decompose` with a small
   prompt that returns JSON steps + deps. Keep the exact `text → Plan`
   interface so nothing downstream changes.
2. **Plan validation.** Before executing, assert the plan produces every
   output the goal implies (catch pitfall #2 *before* runtime).
3. **Cost-aware planning.** Charge tokens per step and have the planner prefer
   a shorter plan when two decompositions are equivalent.
4. **Real parallel executor.** Add `Planner.execute_parallel()` that runs each
   `layers()` layer on a thread pool (fold Section 5 into the module).
5. **Trace every step.** Emit one L72-style span per step (`goal`, `step`,
   `deps`, `tokens`, `replanned`) and query the most-replanned category.

### Bridge to Lesson 81 — Reliability & Partial Failure
Today a failed step got **one** rescue on the generalist. But what if the
generalist fails too? What if a step is *slow* rather than wrong? What if two of
five parallel steps fail and three succeed — do you ship a partial result, retry,
or abort the whole goal? **L81** turns single-shot replanning into a real
**reliability layer**: retries with backoff, timeouts, circuit breakers, and
partial-failure policies — the discipline that makes an orchestrated system
trustworthy in production, heading into the L82 Phase-9 capstone.